In [0]:
%run ../common/environmental_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"

In [0]:
from pyspark.sql import functions as F

In [0]:
df_drivers = spark.read.table(bronze_table)

In [0]:
df_drivers_dropped = df_drivers.drop('url')

In [0]:
df_drivers_renamed = df_drivers_dropped.withColumnsRenamed({
    "driverId": "driver_id",
    "dateOfBirth": "date_of_birth"
})

In [0]:
df_drivers_concat = df_drivers_renamed.withColumn("driver_name" , F.initcap(F.concat_ws(" ", F.col('name.givenname'), F.col('name.familyname'))))

In [0]:
df_drivers_concat = df_drivers_concat.drop('name')

In [0]:
df_drivers_distinct = df_drivers_concat.dropDuplicates(['driver_id'])

In [0]:
df_drivers_final = df_drivers_distinct.withColumn('nationality' , F.initcap('nationality'))

In [0]:
(
    df_drivers_final
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)